# Throughput & Latency vs Concurrent Users
Ramp from 1 → 100 users, measure requests/sec and latency percentiles.

In [ ]:
import threading
import time
import statistics
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from typing import Any, Optional
import matplotlib.pyplot as plt
import random

## Implementations under test

In [ ]:
def run_with_timeout(func, args, timeout):
    """Run a a function with a timeout and return its result or None if it times out or fails.
    This function can be used to set a timeout for tool execution"""
    result = [None]
    exception = [None]
    completed = [False]

    def target():
        try:
            result[0] = func(args)
        except Exception as e:
            exception[0] = e
        finally:
            completed[0] = True

    thread = threading.Thread(target=target)
    thread.daemon = True  # The thread will exit when the main program exits
    thread.start()
    thread.join(timeout)  # applying timeout constraint

    if not completed[0]:  # if it times out
        # print(f"Tool execution timed out after {timeout} seconds")
        return None
    if exception[0]:  # if the tool fails
        # print(f"Tool execution failed: {str(exception[0])}")
        return None
    return result[0]


def run_with_timeout_refactored(func, args, timeout):
    """Fixed implementation — threading.Event, no race."""
    result = [None]
    exception = [None]
    done = threading.Event()

    def target():
        try:
            result[0] = func(args)
        except Exception as e:
            exception[0] = e
        finally:
            done.set()

    thread = threading.Thread(target=target, daemon=True)
    thread.start()

    if not done.wait(timeout):
        return None
    if exception[0] is not None:
        return None
    return result[0]


def run_tools_parallel(
    _rwt,
    ai_msg,
    tools,
    state,
    per_tool_timeout=5,
    parallel_timeout: Optional[float] = None,
    result_timeout: Optional[float] = None,
):
    if not ai_msg.tool_calls:
        return ai_msg.content
    futures = {}
    with ThreadPoolExecutor(max_workers=min(10, len(ai_msg.tool_calls))) as ex:
        for tc in ai_msg.tool_calls:
            sel = next((t for t in tools if t.name == tc["name"]), None)
            if sel is None:
                continue
            futures[ex.submit(_rwt, sel.invoke, tc.get("args", {}), per_tool_timeout)] = tc["name"]
        responses = []
        for f in as_completed(futures, timeout=parallel_timeout):
            try:
                r = f.result(timeout=result_timeout)
                if r is not None:
                    responses.append(r)
            except Exception:
                pass
    return responses if responses else None

## Stubs & harness

In [ ]:
@dataclass
class FakeTool:
    name: str
    func: Any = True
    coroutine: Any = None
    _side_effect: Any = None

    def invoke(self, args):
        return self._side_effect(args) if self._side_effect else f"result_from_{self.name}"


@dataclass
class FakeAIMsg:
    tool_calls: list = field(default_factory=list)
    content: str = ""


def make_tool(name, fn):
    return FakeTool(name=name, _side_effect=fn)


def make_msg(*names):
    return FakeAIMsg(tool_calls=[{"name": n, "args": {}} for n in names])


def simulate_user(uid, tools, msg, per_tool_timeout=5):
    t0 = time.time()
    r = run_tools_parallel(msg, tools, state={}, per_tool_timeout=per_tool_timeout)
    return {"uid": uid, "success": r is not None, "duration_s": time.time() - t0}


def run_load_level(
    _rwt,
    n_users,
    tools_per_user=3,
    per_tool_timeout=5,
    parallel_timeout=5,
    result_timeout=5,
    tool_latency_s=0.1,
    error_rate=0.0,
):

    def make_cfg(uid):
        def tool_fn(args):
            time.sleep(tool_latency_s * random.uniform(0.8, 1.2))
            if random.random() < error_rate:
                raise RuntimeError(f"simulated failure (error_rate={error_rate:.0%})")
            return f"u{args.get('uid')}_result"

        tools = [make_tool(f"t{i}_{uid}", tool_fn) for i in range(tools_per_user)]
        msg = make_msg(*[f"t{i}_{uid}" for i in range(tools_per_user)])
        return tools, msg

    results = [None] * n_users
    lock = threading.Lock()

    def task(uid):
        tools, msg = make_cfg(uid)
        t0 = time.time()
        r = run_tools_parallel(
            _rwt,
            msg,
            tools,
            state={},
            per_tool_timeout=per_tool_timeout,
            parallel_timeout=parallel_timeout,
            result_timeout=result_timeout,
        )
        n_got = len(r) if r else 0
        with lock:
            results[uid] = {
                "uid": uid,
                "success": r is not None,
                "n_got": n_got,
                "n_expected": tools_per_user,
                "n_dropped": tools_per_user - n_got,
                "duration_s": time.time() - t0,
            }

    # plain threads — no outer pool to deadlock with the inner one
    threads = [threading.Thread(target=task, args=(uid,), daemon=True) for uid in range(n_users)]

    wall_start = time.time()
    for t in threads:
        t.start()
    for t in threads:
        t.join(timeout=per_tool_timeout + 10)  # generous but bounded
    wall = time.time() - wall_start

    durations = sorted(r["duration_s"] for r in results if r)
    total_tools = n_users * tools_per_user
    total_dropped = sum(r["n_dropped"] for r in results if r)
    n = len(durations)

    return {
        "n_users": n_users,
        "wall_s": wall,
        "rps": n_users / wall,
        "p50": durations[int(n * 0.50)] * 1000,
        "p90": durations[int(n * 0.90)] * 1000,
        "p95": durations[min(int(n * 0.95), n - 1)] * 1000,
        "p99": durations[min(int(n * 0.99), n - 1)] * 1000,
        "mean": statistics.mean(durations) * 1000,
        "tool_error_rate": total_dropped / total_tools,
        "n_dropped": total_dropped,
        "total_tools": total_tools,
        "req_failures": sum(1 for r in results if r and not r["success"]),
    }


def aggregate(runs, n):
    return {
        "n_users": n,
        "rps_mean": statistics.mean(r["rps"] for r in runs),
        "rps_std": statistics.stdev(r["rps"] for r in runs) if len(runs) > 1 else 0,
        "p50_mean": statistics.mean(r["p50"] for r in runs),
        "p50_std": statistics.stdev(r["p50"] for r in runs) if len(runs) > 1 else 0,
        "p90_mean": statistics.mean(r["p90"] for r in runs),
        "p95_mean": statistics.mean(r["p95"] for r in runs),
        "p99_mean": statistics.mean(r["p99"] for r in runs),
        "p99_std": statistics.stdev(r["p99"] for r in runs) if len(runs) > 1 else 0,
        "tool_error_rate": statistics.mean(r["tool_error_rate"] for r in runs) * 100,
        "req_failures": statistics.mean(r["req_failures"] for r in runs),
    }

## Run the ramp

In [ ]:
LEVELS = [2, 4, 8, 16, 32, 64, 128, 256, 512]
TOOLS_PER_USER = 5  # tools each user invokes in parallel
N_REPEATS = 10  # runs per level — increase for more stable results
TOOL_LATENCY_S = 0.001
TOOL_ERROR_RATE = 0.01
TOOL_TIMEOUT = 0.01
PARALLEL_TIMEOUT = None
RESULT_TIMEOUT = None

print(
    f"{'Users':>6}  {'RPS mean':>9}  {'RPS std':>8}  {'p50 mean':>9}  {'p99 mean':>9}  {'Tool Err%':>10}  {'Req Fails':>10}"
)
print("-" * 80)

stats_bad = []
stats_fixed = []
for n in LEVELS:
    runs_original = [
        run_load_level(
            run_with_timeout,
            n,
            tools_per_user=TOOLS_PER_USER,
            per_tool_timeout=TOOL_TIMEOUT,
            parallel_timeout=PARALLEL_TIMEOUT,
            result_timeout=RESULT_TIMEOUT,
            tool_latency_s=TOOL_LATENCY_S,
            error_rate=TOOL_ERROR_RATE,
        )
        for _ in range(N_REPEATS)
    ]
    runs_refactored = [
        run_load_level(
            run_with_timeout_refactored,
            n,
            tools_per_user=TOOLS_PER_USER,
            per_tool_timeout=TOOL_TIMEOUT,
            parallel_timeout=PARALLEL_TIMEOUT,
            result_timeout=RESULT_TIMEOUT,
            tool_latency_s=TOOL_LATENCY_S,
            error_rate=TOOL_ERROR_RATE,
        )
        for _ in range(N_REPEATS)
    ]
    stats_bad.append(aggregate(runs_original, n))
    stats_fixed.append(aggregate(runs_refactored, n))

    b, f = stats_bad[-1], stats_fixed[-1]
    print(
        f"{n:>6}  original  {b['rps_mean']:>9.1f}  {b['rps_std']:>8.1f}  "
        f"{b['p50_mean']:>8.0f}ms  {b['p99_mean']:>8.0f}ms  "
        f"{b['tool_error_rate']:>9.1f}%  {b['req_failures']:>10.1f}"
    )
    print(
        f"{'':>6}  refactored  {f['rps_mean']:>9.1f}  {f['rps_std']:>8.1f}  "
        f"{f['p50_mean']:>8.0f}ms  {f['p99_mean']:>8.0f}ms  "
        f"{f['tool_error_rate']:>9.1f}%  {f['req_failures']:>10.1f}"
    )
    print()

## Plot

In [ ]:
ERR_COLOR = "#f85149"
ORIGINAL_COLOR = "#e67e22"  # orange — original
REFACTORED_COLOR = "#58a6ff"  # blue   — refactored

ns = [s["n_users"] for s in stats_bad]  # same levels for both

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Tool Runner Load Test - n={N_REPEATS} runs/level", fontsize=13, y=1.01)

for stats, label, color in [
    (stats_bad, "original", ORIGINAL_COLOR),
    (stats_fixed, "refactored", REFACTORED_COLOR),
]:
    rps_mean = [s["rps_mean"] for s in stats]
    rps_std = [s["rps_std"] for s in stats]
    p50_mean = [s["p50_mean"] for s in stats]
    p99_mean = [s["p99_mean"] for s in stats]
    p50_std = [s["p50_std"] for s in stats]
    p99_std = [s["p99_std"] for s in stats]

    rps_lo = [m - s for m, s in zip(rps_mean, rps_std)]
    rps_hi = [m + s for m, s in zip(rps_mean, rps_std)]
    p50_lo = [m - s for m, s in zip(p50_mean, p50_std)]
    p50_hi = [m + s for m, s in zip(p50_mean, p50_std)]
    p99_lo = [m - s for m, s in zip(p99_mean, p99_std)]
    p99_hi = [m + s for m, s in zip(p99_mean, p99_std)]

    # ── throughput ────────────────────────────────────────────────────────────
    ax1.plot(ns, rps_mean, "o-", color=color, lw=2.5, ms=7, label=label, zorder=3)
    ax1.fill_between(ns, rps_lo, rps_hi, alpha=0.15, color=color)

    # ── latency ───────────────────────────────────────────────────────────────
    ax2.plot(ns, p50_mean, "o-", color=color, lw=2, ms=6, label=f"{label} p50")
    ax2.plot(ns, p99_mean, "o--", color=color, lw=1.5, ms=4, label=f"{label} p99", alpha=0.6)
    ax2.fill_between(ns, p50_lo, p50_hi, alpha=0.1, color=color)
    ax2.fill_between(ns, p99_lo, p99_hi, alpha=0.1, color=color)
    ax2.fill_between(ns, p50_mean, p99_mean, alpha=0.05, color=color)

ax1.set_xlabel("Concurrent users")
ax1.set_ylabel("Requests / second")
ax1.set_title("Throughput")
ax1.set_xticks(ns)
ax1.legend()
ax1.grid(True, alpha=0.5)

ax2.set_xlabel("Concurrent users")
ax2.set_ylabel("Latency (ms)")
ax2.set_title("Latency")
ax2.set_xticks(ns)
ax2.legend()
ax2.grid(True, alpha=0.5)

plt.tight_layout()
plt.show()